# Score Differences

Where there any conversations where Maverick scored higher after we added Alexandria content?

In [ ]:
import polars as pl
from sklearn.model_selection import train_test_split

from simple_evals.improvement.models.benchmark_inputs import EvalInput
from simple_evals.improvement.paths import EVAL_INPUTS

In [4]:
eval_input_dicts = [i.flatten() for i in EvalInput.from_inputs(EVAL_INPUTS)]
eval_inputs = pl.DataFrame(eval_input_dicts)
eval_inputs


prompt_id,theme
str,str
"""1f548d5b-cd00-49a0-b327-283a2e…","""context_seeking"""
"""0b8f1d60-2081-4562-98f7-b6a976…","""communication"""
"""6f7a2ee9-e9c6-42d8-b79f-22dea9…","""emergency_referrals"""
"""19ec4833-86e9-4166-8b82-d1da09…","""hedging"""
"""7ebc830a-8dbd-489b-9d61-4d8bac…","""emergency_referrals"""
…,…
"""c4d8b028-f148-47ca-b1bc-c5c467…","""health_data_tasks"""
"""0175eaea-4d14-4932-956b-601951…","""communication"""
"""a644d818-6989-42b4-84b7-f87ff1…","""communication"""


In [ ]:
# Create the train/test splits
train, test = train_test_split(
    eval_inputs["prompt_id"].to_numpy(),
    train_size=0.5,
    random_state=42,
    stratify=eval_inputs["theme"].to_numpy(),
)
train_df = pl.DataFrame({"prompt_id": train, "train_test": "train"})
test_df = pl.DataFrame({"prompt_id": test, "train_test": "test"})
# Join the split information back into the list of prompt_id's
with_train_test = eval_inputs.join(
    pl.concat([train_df, test_df]), on="prompt_id", how="left"
)
# Check that the classes are roughly equal in the two splits
counts = with_train_test.group_by("theme", "train_test").len().group_by("theme")
counts.head()


theme,train_test,len
str,str,u32
"""complex_responses""","""train""",180
"""complex_responses""","""test""",180
"""context_seeking""","""train""",297
"""context_seeking""","""test""",297
"""communication""","""train""",459
…,…,…
"""hedging""","""test""",535
"""emergency_referrals""","""test""",241
"""emergency_referrals""","""train""",241


In [24]:
# Write out the train-test splits to a file
with_train_test.write_csv("train_test.csv")